In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [6]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

smart_mcq_solver_challenge_path = kagglehub.competition_download('smart-mcq-solver-challenge')

print('Data source import complete.')


Data source import complete.


In [10]:
!cp -r /root/.cache/kagglehub/competitions/ .

In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
train_df = pd.read_csv('/content/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/content/competitions/smart-mcq-solver-challenge/test.csv')
sample_df = pd.read_csv('/content/competitions/smart-mcq-solver-challenge/sample_submission.csv')

## Questions

#### 1. Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [12]:
counts = train_df['answer'].value_counts()
most_frequent = counts.max()
least_frequent = counts.min()
sum_frequencies = most_frequent + least_frequent

print("Frequency distribution:")
print(counts)
print(f"\nMost frequent count: {most_frequent}")
print(f"Least frequent count: {least_frequent}")
print(f"Sum of most and least frequent: {sum_frequencies}")

Frequency distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent count: 490
Least frequent count: 324
Sum of most and least frequent: 814


#### After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [13]:
import string

# Create translation table for punctuation
punct_table = str.maketrans('', '', string.punctuation)

# Efficiently clean and extract unique words using a set comprehension
# We process the 'prompt' column: lowercase, remove punctuation, and split by whitespace
vocab = {
    word
    for prompt in train_df['prompt'].dropna()
    for word in str(prompt).lower().translate(punct_table).split()
}

print(f"Total number of unique words (Vocabulary size): {len(vocab):,}")

Total number of unique words (Vocabulary size): 859


#### Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

In [14]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

cleaned_prompt1 = train_df['prompt'][0].lower().translate(punct_table).split()
filtered_prompt1 = [w for w in cleaned_prompt1 if w not in ENGLISH_STOP_WORDS and w != '']

print(filtered_prompt1.__len__())

13


#### Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer with English stop words
vectorizer = TfidfVectorizer(stop_words='english')

# Combine prompt and all options into a single corpus
columns = ['prompt', 'A', 'B', 'C', 'D', 'E']
combined_text = train_df[columns].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1)

# Fit the vectorizer and transform the data
vect_matrix = vectorizer.fit_transform(combined_text)

# Get the total number of feature columns (vocabulary size)
vocab_size = len(vectorizer.get_feature_names_out())

print(f"Total number of feature columns: {vocab_size}")

Total number of feature columns: 2762


#### Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

prompt_vector = vectorizer.transform(train_df['prompt'])
option_a_vector = vectorizer.transform(train_df['A'])

similarity_score = cosine_similarity(prompt_vector, option_a_vector)

print(f"{similarity_score[0][0]:.4f}")

0.2720


#### Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def compute_cosine_sim(v1, v2):
    # Since TfidfVectorizer outputs L2-normalized vectors,
    # cosine similarity is simply the dot product (element-wise mult + sum)
    return np.array(v1.multiply(v2).sum(axis=1)).flatten()

# 1. Prepare data
df = train_df.copy()
options = ['A', 'B', 'C', 'D', 'E']
for col in ['prompt'] + options:
    df[col] = df[col].fillna('').astype(str)

# 2. Re-fit vectorizer on full context (as per Question 3 setup)
vectorizer = TfidfVectorizer(stop_words='english')
full_text = df[['prompt'] + options].apply(lambda x: ' '.join(x), axis=1)
vectorizer.fit(full_text)

# 3. Transform components
p_vecs = vectorizer.transform(df['prompt'])

# 4. Calculate similarities for each option across all rows
sim_results = {}
for opt in options:
    opt_vecs = vectorizer.transform(df[opt])
    sim_results[opt] = compute_cosine_sim(p_vecs, opt_vecs)

# 5. Analyze results
sim_df = pd.DataFrame(sim_results)
df['highest_sim_option'] = sim_df.idxmax(axis=1)

# 6. Calculate accuracy percentage
accuracy = (df['highest_sim_option'] == df['answer']).mean() * 100

print(f"Percentage of instances where highest similarity matches the answer: {accuracy:.2f}%")

Percentage of instances where highest similarity matches the answer: 13.15%


#### If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [19]:
def map_at_3(actual, predicted):
    try:
        index = predicted.index(actual)
        return 1 / (index + 1)
    except ValueError:
        return 0

ground_truth = 'C'
predictions = ['C', 'A', 'B']
score = map_at_3(ground_truth, predictions)

print(f"MAP@3 Score: {score:.1f}")

MAP@3 Score: 1.0


#### If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?

In [20]:
ground_truth_2 = 'B'
predictions_2 = ['D', 'B', 'E']
score_2 = map_at_3(ground_truth_2, predictions_2)

print(f"MAP@3 Score: {score_2:.1f}")

MAP@3 Score: 0.5


#### The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [21]:
# 1. Get the top 3 most frequent answers from Question 1
top_3_baseline = train_df['answer'].value_counts().index[:3].tolist()

# 2. Define the MAP@3 function
def map_at_3(actual, predicted):
    try:
        index = predicted.index(actual)
        return 1 / (index + 1)
    except ValueError:
        return 0

# 3. Apply the baseline prediction to every row and calculate the average score
train_df['baseline_score'] = train_df['answer'].apply(lambda x: map_at_3(x, top_3_baseline))
overall_map3 = train_df['baseline_score'].mean()

print(f"Top 3 Baseline Predictions: {top_3_baseline}")
print(f"Overall MAP@3 score of Majority Class baseline: {overall_map3:.4f}")

Top 3 Baseline Predictions: ['B', 'C', 'A']
Overall MAP@3 score of Majority Class baseline: 0.4213


#### The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

In [22]:
# 1. Reuse the sim_df (similarities for A, B, C, D, E) calculated in Question 5
# 2. Get the top 3 options for each row by sorting similarity values descending
def get_top_3(row):
    return row.sort_values(ascending=False).index[:3].tolist()

train_df['tfidf_top_3'] = sim_df.apply(get_top_3, axis=1)

# 3. Calculate MAP@3 for each row using the existing map_at_3 function
train_df['tfidf_map3_score'] = train_df.apply(
    lambda row: map_at_3(row['answer'], row['tfidf_top_3']),
    axis=1
)

# 4. Calculate the overall average
final_avg_map3 = train_df['tfidf_map3_score'].mean()

print(f"Average MAP@3 score for the TF-IDF pipeline: {final_avg_map3:.4f}")

Average MAP@3 score for the TF-IDF pipeline: 0.2937
